# 基于ShuffleNet的山火识别
## 1.加载overlay

In [2]:
import time
from pynq import Overlay
import time
import numpy as np
from pynq import Xlnk
import struct
from scipy.misc import imread
import cv2
import os

overlay = Overlay('./shufflenet.bit')
print("Overlay downloaded successfully!")

Overlay downloaded successfully!


## 2.定义IP核驱动及其他功能函数

In [3]:
xlnk = Xlnk()


def sw_conv_bn_relu_layer(feature_in, conv_kernel, bn_mean, bn_var, bn_gamma, bn_beta,
                  kernel_size, stride, padding, conv_type):
    """
    卷积+BN层的Python实现（使用标准numpy数组布局）
    
    参数:
        feature_in: 输入特征图，形状 (H_in, W_in, C_in)
        conv_kernel: 卷积核权重
            - 普通卷积: (C_out, C_in, kernel_size, kernel_size)
            - 深度卷积: (C_out, kernel_size, kernel_size)
        bn_mean: BN均值，形状 (C_out,)
        bn_var: BN方差，形状 (C_out,)
        bn_gamma: BN缩放参数，形状 (C_out,)
        bn_beta: BN偏移参数，形状 (C_out,)
        kernel_size: 卷积核大小
        stride: 步长
        padding: 填充
        conv_type: 卷积类型 ('NORMAL_CONV' 或 'DEPTHWISE_CONV')
    
    返回:
        feature_out: 输出特征图，形状 (H_out, W_out, C_out)
    """
    H_in, W_in, C_in = feature_in.shape
    # 计算输出尺寸
    H_out = (H_in + 2 * padding - kernel_size) // stride + 1
    W_out = (W_in + 2 * padding - kernel_size) // stride + 1
    C_out = len(bn_mean)
    # 初始化输出特征图
    feature_out = np.zeros((H_out, W_out, C_out), dtype=np.float32)
    # 对输入进行padding
    if padding > 0:
        feature_in_padded = np.pad(feature_in, 
                                   ((padding, padding), (padding, padding), (0, 0)),
                                   mode='constant', constant_values=0)
    else:
        feature_in_padded = feature_in
    if conv_type == 0:
        # 普通卷积实现
        # conv_kernel形状: (C_out, C_in, kernel_size, kernel_size)
        
        for h_out in range(H_out):
            for w_out in range(W_out):
                # 计算输入窗口的起始位置
                h_start = h_out * stride
                w_start = w_out * stride
                # 提取输入窗口 (kernel_size, kernel_size, C_in)
                input_window = feature_in_padded[h_start:h_start+kernel_size, w_start:w_start+kernel_size, :]
                
                for c_out in range(C_out):
                    sum_val = 0.0
                    for c_in in range(C_in):
                        for kh in range(kernel_size):
                            for kw in range(kernel_size):
                                sum_val += (input_window[kh, kw, c_in] * conv_kernel[c_out, c_in, kh, kw])
                    
                    # Batch Normalization计算
                    eps = 1e-5
                    val = sum_val - bn_mean[c_out]
                    val = val * bn_gamma[c_out] / np.sqrt(bn_var[c_out] + eps)
                    val += bn_beta[c_out]
                    
                    # 量化并应用ReLU激活
                    bn_val = val
                    bn_val = max(0, bn_val)  # ReLU
                    
                    feature_out[h_out, w_out, c_out] = bn_val
    
    elif conv_type == 1:
        # 深度卷积实现
        # conv_kernel形状: (C_out, kernel_size, kernel_size)
        
        for h_out in range(H_out):
            for w_out in range(W_out):
                # 计算输入窗口的起始位置
                h_start = h_out * stride
                w_start = w_out * stride
                
                # 提取输入窗口 (kernel_size, kernel_size, C_in)
                input_window = feature_in_padded[h_start:h_start+kernel_size, w_start:w_start+kernel_size, :]
                
                for c_out in range(C_out):
                    # 深度卷积：每个通道独立处理
                    c_in = c_out
                    
                    sum_val = 0.0
                    for kh in range(kernel_size):
                        for kw in range(kernel_size):
                            sum_val += (input_window[kh, kw, c_in] * conv_kernel[c_out, kh, kw])
                    
                    # Batch Normalization计算
                    eps = 1e-5
                    val = sum_val - bn_mean[c_out]
                    val = val * bn_gamma[c_out] / np.sqrt(bn_var[c_out] + eps)
                    val += bn_beta[c_out]
                    # 量化（深度卷积不需要激活）
                    bn_val = val
                    feature_out[h_out, w_out, c_out] = bn_val
    
    return feature_out

def sw_pooling(feature_in, C_in, H_in, W_in, kernel_size, stride, padding, pool_type):
    """
    池化层的Python实现
    
    参数:
        feature_in: 输入特征图，形状 (H_in, W_in, C_in)
        C_in: 输入通道数
        H_in: 输入高度
        W_in: 输入宽度
        kernel_size: 池化核大小
        stride: 池化步长
        padding: 池化填充
        pool_type: 池化类型 (0: MAX_POOL, 1: AVG_POOL)
    
    返回:
        feature_out: 输出特征图，形状 (H_out, W_out, C_in)
    """
    # 计算输出尺寸
    H_out = (H_in + 2 * padding - kernel_size) // stride + 1
    W_out = (W_in + 2 * padding - kernel_size) // stride + 1
    
    # 初始化输出特征图
    feature_out = np.zeros((H_out, W_out, C_in), dtype=feature_in.dtype)
    
    # 遍历每个通道
    for c in range(C_in):
        # 遍历输出空间位置
        for h_out in range(H_out):
            for w_out in range(W_out):
                if pool_type == 0:  # 最大池化
                    # 初始化最大值为最小值
                    max_val = -128
                    
                    # 遍历池化窗口
                    for kh in range(kernel_size):
                        for kw in range(kernel_size):
                            # 计算输入坐标
                            h_in = h_out * stride + kh - padding
                            w_in = w_out * stride + kw - padding
                            
                            # 边界检查
                            if h_in >= 0 and h_in < H_in and w_in >= 0 and w_in < W_in:
                                # 更新最大值
                                if feature_in[h_in, w_in, c] > max_val:
                                    max_val = feature_in[h_in, w_in, c]
                    
                    # 存储最大池化结果
                    feature_out[h_out, w_out, c] = max_val
                    
                else:  # 平均池化
                    # 初始化累加器和有效像素计数
                    sum_val = 0
                    count = 0
                    
                    # 遍历池化窗口
                    for kh in range(kernel_size):
                        for kw in range(kernel_size):
                            # 计算输入坐标
                            h_in = h_out * stride + kh - padding
                            w_in = w_out * stride + kw - padding
                            
                            # 边界检查
                            if h_in >= 0 and h_in < H_in and w_in >= 0 and w_in < W_in:
                                # 累加值并增加计数
                                sum_val += feature_in[h_in, w_in, c]
                                count += 1
                    
                    # 计算平均值并存储结果
                    avg_val = (sum_val / count) if count > 0 else 0
                    feature_out[h_out, w_out, c] = avg_val
    
    return feature_out
def sw_channel_shuffle(in_feature, c_in, h_w, groups):
    """
    通道混洗的Python实现
    
    参数:
        in_feature: 输入特征图，形状 (h_w, h_w, c_in)
        c_in: 输入通道数
        h_w: 特征图高度和宽度
        groups: 分组数量
    
    返回:
        out_feature: 输出特征图，形状 (h_w, h_w, c_in)
    """
    
    # 初始化输出特征图
    out_feature = np.zeros((h_w, h_w, c_in), dtype=in_feature.dtype)
    
    # 计算每组的通道数
    group_ch = c_in // groups
    
    # 遍历空间位置
    for h in range(h_w):
        for w in range(h_w):
            # 循环拆分以减少循环复杂度
            for g in range(groups):
                for c in range(group_ch):
                    # 计算输入索引
                    in_c = g * group_ch + c
                    
                    # 跨组重组：将每个组的第c个通道放到第c组的第g个位置
                    out_c = c * groups + g
                    
                    # 执行通道混洗操作
                    out_feature[h, w, out_c] = in_feature[h, w, in_c]
    
    return out_feature

# def sw_fully_connected(in_features, out_features, input_vec, weights, biases):
#     """
#     全连接层的Python实现
    
#     参数:
#         in_features: 输入特征数量
#         out_features: 输出特征数量
#         input_vec: 输入特征向量，形状 (in_features,)
#         weights: 权重矩阵，形状 (out_features, in_features)
#         biases: 偏置向量，形状 (out_features,)
    
#     返回:
#         output: 输出结果向量，形状 (out_features,)
#     """
    
#     # 初始化输出向量
#     output = np.zeros(out_features, dtype=input_vec.dtype)
    
#     # 为每个输出特征计算全连接结果
#     for o in range(out_features):
#         # 初始化累加器
#         sum_val = 0
#         # 计算加权和
#         for i in range(in_features):
#             # 计算权重索引（行优先存储）
#             w_offset = o * in_features + i
            
#             # 累加加权和
#             sum_val += input_vec[i] * weights[w_offset]
        
#         # 添加偏置
#         sum_val += biases[o]
        
#         # 直接输出结果（无激活函数）
#         output[o] = sum_val
    
#     return output
def sw_fully_connected(in_features, out_features, input_vec, weights, biases):
    """
    全连接层的Python实现（修正二维权重索引）
    
    参数:
        in_features: 输入特征数量
        out_features: 输出特征数量
        input_vec: 输入特征向量，形状 (in_features,)
        weights: 权重矩阵，形状 (out_features, in_features) （二维数组）
        biases: 偏置向量，形状 (out_features,)
    
    返回:
        output: 输出结果向量，形状 (out_features,)
    """
    output = np.zeros(out_features, dtype=input_vec.dtype)
    
    for o in range(out_features):  # o：输出神经元索引（0~3，对应权重的行）
        sum_val = 0
        for i in range(in_features):  # i：输入特征索引（0~1023，对应权重的列）
            # 直接用二维索引访问：第o行第i列的权重，无需计算w_offset
            sum_val += input_vec[i] * weights[o][i]  # 核心修改点
        sum_val += biases[o]
        output[o] = sum_val
    
    return output
def readbinfile(filename,size):
    f = open(filename, "rb")
    z=[]
    for j in range(size):
        data = f.read(4)
        data_float = struct.unpack("f", data)[0]
        z.append(data_float)
    f.close()
    z = np.array(z)
    return z 
print("step 2 done.")

step 2 done.


## 3.读取网络参数

In [4]:
#输入iamge
image = np.zeros((128, 128, 3), dtype=np.float32)
#####################################################
#Conv1卷积核，BN参数
W_conv1 = np.zeros((24, 3, 3, 3), dtype=np.float32)
W_conv1 = readbinfile("./data/conv1_0_weight.bin", 24*3*3*3).reshape((24, 3, 3, 3))
BN_mean_conv1 = readbinfile("./data/conv1_1_running_mean.bin", 24)
BN_val_conv1 = readbinfile("./data/conv1_1_running_var.bin", 24)
BN_gamma_conv1 = readbinfile("./data/conv1_1_weight.bin", 24)
BN_beta_conv1 = readbinfile("./data/conv1_1_bias.bin", 24)
#Conv1输出64X64x24
out_conv1 = np.zeros(( 64, 64, 24), dtype=np.float32)
#Maxpool输出32x32x24
out_maxpool = np.zeros((32, 32, 24), dtype=np.float32)
#####################################################
#stage2下采样单元-分支1-3x3深度卷积输出
out_s2s_b1_convdw = np.zeros((16, 16, 24), dtype=np.float32)
#stage2下采样单元-分支1-3x3深度卷积核，BN参数
s2s_b1_w_convdw = np.zeros((24, 3, 3), dtype=np.float32)
S2s_b1_w_convdw = readbinfile("./data/stage2_0_branch1_0_weight.bin", 24*3*3)
S2s_b1_w_convdw = S2s_b1_w_convdw.reshape((24, 3, 3))
S2s_b1_bn_mean_convdw = readbinfile("./data/stage2_0_branch1_1_running_mean.bin", 24)
S2s_b1_bn_val_convdw = readbinfile("./data/stage2_0_branch1_1_running_var.bin", 24)
S2s_b1_bn_gamma_convdw = readbinfile("./data/stage2_0_branch1_1_weight.bin", 24)
S2s_b1_bn_beta_convdw = readbinfile("./data/stage2_0_branch1_1_bias.bin", 24)
#stage2下采样单元-分支1-1x1普通卷积输出
out_s2s_b1_conv1 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2下采样单元-分支1-1x1普通卷积核，BN参数
s2s_b1_w_conv1 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2s_b1_w_conv1 = readbinfile("./data/stage2_0_branch1_2_weight.bin", 24*24*1*1)
S2s_b1_w_conv1 = S2s_b1_w_conv1.reshape((24, 24, 1, 1))
S2s_b1_bn_mean_conv1 = readbinfile("./data/stage2_0_branch1_3_running_mean.bin", 24)
S2s_b1_bn_val_conv1 = readbinfile("./data/stage2_0_branch1_3_running_var.bin", 24)
S2s_b1_bn_gamma_conv1 = readbinfile("./data/stage2_0_branch1_3_weight.bin", 24)
S2s_b1_bn_beta_conv1 = readbinfile("./data/stage2_0_branch1_3_bias.bin", 24)
#stage2下采样单元-分支2-1x1普通卷积1输出
out_s2s_b2_conv1 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2下采样单元-分支2-1x1普通卷积1核，BN参数
S2s_b2_w_conv1 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2s_b2_w_conv1 = readbinfile("./data/stage2_0_branch2_0_weight.bin", 24*24*1*1)
S2s_b2_w_conv1 = S2s_b2_w_conv1.reshape((24, 24, 1, 1))
S2s_b2_bn_mean_conv1 = readbinfile("./data/stage2_0_branch2_1_running_mean.bin", 24)
S2s_b2_bn_val_conv1 = readbinfile("./data/stage2_0_branch2_1_running_var.bin", 24)
S2s_b2_bn_gamma_conv1 = readbinfile("./data/stage2_0_branch2_1_weight.bin", 24)
S2s_b2_bn_beta_conv1 = readbinfile("./data/stage2_0_branch2_1_bias.bin", 24)
#stage2下采样单元-分支2-3x3深度卷积输出
out_s2s_b2_convdw = np.zeros((16, 16, 24), dtype=np.float32)
#stage2下采样单元-分支2-3x3深度卷积核，BN参数
s2s_b2_w_convdw = np.zeros((24, 3, 3), dtype=np.float32)
S2s_b2_w_convdw = readbinfile("./data/stage2_0_branch2_3_weight.bin", 24*3*3)
S2s_b2_w_convdw = S2s_b2_w_convdw.reshape((24, 3, 3))
S2s_b2_bn_mean_convdw = readbinfile("./data/stage2_0_branch2_4_running_mean.bin", 24)
S2s_b2_bn_val_convdw = readbinfile("./data/stage2_0_branch2_4_running_var.bin", 24)
S2s_b2_bn_gamma_convdw = readbinfile("./data/stage2_0_branch2_4_weight.bin", 24)
S2s_b2_bn_beta_convdw = readbinfile("./data/stage2_0_branch2_4_bias.bin", 24)
#stage2下采样单元-分支2-1x1普通卷积输出2
out_s2s_b2_conv2 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2下采样单元-分支2-1x1普通卷积核2，BN参数
S2s_b2_w_conv2 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2s_b2_w_conv2 = readbinfile("./data/stage2_0_branch2_5_weight.bin", 24*24*1*1)
S2s_b2_w_conv2 = S2s_b2_w_conv2.reshape((24, 24, 1, 1))
S2s_b2_bn_mean_conv2 = readbinfile("./data/stage2_0_branch2_6_running_mean.bin", 24)
S2s_b2_bn_val_conv2 = readbinfile("./data/stage2_0_branch2_6_running_var.bin", 24)
S2s_b2_bn_gamma_conv2 = readbinfile("./data/stage2_0_branch2_6_weight.bin", 24)
S2s_b2_bn_beta_conv2 = readbinfile("./data/stage2_0_branch2_6_bias.bin", 24)
#stage2下采样单元-通道合并，通道混洗
in_s2s_shuff = np.zeros((16, 16, 48), dtype=np.float32)
out_s2s_shuff = np.zeros((16, 16, 48), dtype=np.float32)

#基本单元1
#stage2基本单元1-通道拆分
out_s2c1_ch_spilt1 = np.zeros((16, 16, 24), dtype=np.float32)
out_s2c1_ch_spilt2 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元1-分支1
#stage2基本单元1-分支2-1x1普通卷积1输出
out_s2c1_b2_conv1 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元1-分支2-1x1普通卷积1核，BN参数
s2c1_b2_w_conv1 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2c1_b2_w_conv1 = readbinfile("./data/stage2_1_branch2_0_weight.bin", 24*24*1*1)
S2c1_b2_w_conv1 = S2c1_b2_w_conv1.reshape((24, 24, 1, 1))
S2c1_b2_bn_mean_conv1 = readbinfile("./data/stage2_1_branch2_1_running_mean.bin", 24)
S2c1_b2_bn_val_conv1 = readbinfile("./data/stage2_1_branch2_1_running_var.bin", 24)
S2c1_b2_bn_gamma_conv1 = readbinfile("./data/stage2_1_branch2_1_weight.bin", 24)
S2c1_b2_bn_beta_conv1 = readbinfile("./data/stage2_1_branch2_1_bias.bin", 24)
#stage2基本单元1-分支2-3x3深度卷积输出
out_s2c1_b2_convdw = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元1-分支2-3x3深度卷积核，BN参数
s2c1_b2_w_convdw = np.zeros((24, 3, 3), dtype=np.float32)
S2c1_b2_w_convdw = readbinfile("./data/stage2_1_branch2_3_weight.bin", 24*3*3)
S2c1_b2_w_convdw = S2c1_b2_w_convdw.reshape((24, 3, 3))
S2c1_b2_bn_mean_convdw = readbinfile("./data/stage2_1_branch2_4_running_mean.bin", 24)
S2c1_b2_bn_val_convdw = readbinfile("./data/stage2_1_branch2_4_running_var.bin", 24)
S2c1_b2_bn_gamma_convdw = readbinfile("./data/stage2_1_branch2_4_weight.bin", 24)
S2c1_b2_bn_beta_convdw = readbinfile("./data/stage2_1_branch2_4_bias.bin", 24)
#stage2基本单元1-分支2-1x1普通卷积输出2
out_s2c1_b2_conv2 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元1-分支2-1x1普通卷积核2，BN参数
s2c1_b2_w_conv2 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2c1_b2_w_conv2 = readbinfile("./data/stage2_1_branch2_5_weight.bin", 24*24*1*1)
S2c1_b2_w_conv2 = S2c1_b2_w_conv2.reshape((24, 24, 1, 1))
S2c1_b2_bn_mean_conv2 = readbinfile("./data/stage2_1_branch2_6_running_mean.bin", 24)
S2c1_b2_bn_val_conv2 = readbinfile("./data/stage2_1_branch2_6_running_var.bin", 24)
S2c1_b2_bn_gamma_conv2 = readbinfile("./data/stage2_1_branch2_6_weight.bin", 24)
S2c1_b2_bn_beta_conv2 = readbinfile("./data/stage2_1_branch2_6_bias.bin", 24)
#stage2基本单元1-通道合并，通道混洗
in_s2c1_shuff = np.zeros((16, 16, 48), dtype=np.float32)
out_s2c1_shuff = np.zeros((16, 16, 48), dtype=np.float32)

#基本单元2
#stage2基本单元2-通道拆分
out_s2c2_ch_spilt1 = np.zeros((16, 16, 24), dtype=np.float32)
out_s2c2_ch_spilt2 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元2-分支1
#stage2基本单元2-分支2-1x1普通卷积1输出
out_s2c2_b2_conv1 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元2-分支2-1x1普通卷积1核，BN参数
s2c2_b2_w_conv1 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2c2_b2_w_conv1 = readbinfile("./data/stage2_2_branch2_0_weight.bin", 24*24*1*1)
S2c2_b2_w_conv1 = S2c2_b2_w_conv1.reshape((24, 24, 1, 1))
S2c2_b2_bn_mean_conv1 = readbinfile("./data/stage2_2_branch2_1_running_mean.bin", 24)
S2c2_b2_bn_val_conv1 = readbinfile("./data/stage2_2_branch2_1_running_var.bin", 24)
S2c2_b2_bn_gamma_conv1 = readbinfile("./data/stage2_2_branch2_1_weight.bin", 24)
S2c2_b2_bn_beta_conv1 = readbinfile("./data/stage2_2_branch2_1_bias.bin", 24)
#stage2基本单元2-分支2-3x3深度卷积输出
out_s2c2_b2_convdw = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元2-分支2-3x3深度卷积核，BN参数
s2c2_b2_w_convdw = np.zeros((24, 3, 3), dtype=np.float32)
S2c2_b2_w_convdw = readbinfile("./data/stage2_2_branch2_3_weight.bin", 24*3*3).reshape((24, 3, 3))
S2c2_b2_bn_mean_convdw = readbinfile("./data/stage2_2_branch2_4_running_mean.bin", 24)
S2c2_b2_bn_val_convdw = readbinfile("./data/stage2_2_branch2_4_running_var.bin", 24)
S2c2_b2_bn_gamma_convdw = readbinfile("./data/stage2_2_branch2_4_weight.bin", 24)
S2c2_b2_bn_beta_convdw = readbinfile("./data/stage2_2_branch2_4_bias.bin", 24)
#stage2基本单元2-分支2-1x1普通卷积输出2
out_s2c2_b2_conv2 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元2-分支2-1x1普通卷积核2，BN参数
s2c2_b2_w_conv2 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2c2_b2_w_conv2 = readbinfile("./data/stage2_2_branch2_5_weight.bin", 24*24*1*1).reshape((24, 24, 1, 1))
S2c2_b2_bn_mean_conv2 = readbinfile("./data/stage2_2_branch2_6_running_mean.bin", 24)
S2c2_b2_bn_val_conv2 = readbinfile("./data/stage2_2_branch2_6_running_var.bin", 24)
S2c2_b2_bn_gamma_conv2 = readbinfile("./data/stage2_2_branch2_6_weight.bin", 24)
S2c2_b2_bn_beta_conv2 = readbinfile("./data/stage2_2_branch2_6_bias.bin", 24)
#stage2基本单元2-通道合并，通道混洗
in_s2c2_shuff = np.zeros((16, 16, 48), dtype=np.float32)
out_s2c2_shuff = np.zeros((16, 16, 48), dtype=np.float32)

#基本单元3
#stage2基本单元3-通道拆分
out_s2c3_ch_spilt1 = np.zeros((16, 16, 24), dtype=np.float32)
out_s2c3_ch_spilt2 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元3-分支1
#stage2基本单元3-分支2-1x1普通卷积1输出
out_s2c3_b2_conv1 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元3-分支2-1x1普通卷积1核，BN参数
s2c3_b2_w_conv1 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2c3_b2_w_conv1 = readbinfile("./data/stage2_3_branch2_0_weight.bin", 24*24*1*1).reshape((24, 24, 1, 1))
S2c3_b2_bn_mean_conv1 = readbinfile("./data/stage2_3_branch2_1_running_mean.bin", 24)
S2c3_b2_bn_val_conv1 = readbinfile("./data/stage2_3_branch2_1_running_var.bin", 24)
S2c3_b2_bn_gamma_conv1 = readbinfile("./data/stage2_3_branch2_1_weight.bin", 24)
S2c3_b2_bn_beta_conv1 = readbinfile("./data/stage2_3_branch2_1_bias.bin", 24)
#stage2基本单元3-分支2-3x3深度卷积输出
out_s2c3_b2_convdw = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元3-分支2-3x3深度卷积核，BN参数
s2c3_b2_w_convdw = np.zeros((24, 3, 3), dtype=np.float32)
S2c3_b2_w_convdw = readbinfile("./data/stage2_3_branch2_3_weight.bin", 24*3*3).reshape((24, 3, 3))
S2c3_b2_bn_mean_convdw = readbinfile("./data/stage2_3_branch2_4_running_mean.bin", 24)
S2c3_b2_bn_val_convdw = readbinfile("./data/stage2_3_branch2_4_running_var.bin", 24)
S2c3_b2_bn_gamma_convdw = readbinfile("./data/stage2_3_branch2_4_weight.bin", 24)
S2c3_b2_bn_beta_convdw = readbinfile("./data/stage2_3_branch2_4_bias.bin", 24)
#stage2基本单元3-分支2-1x1普通卷积输出2
out_s2c3_b2_conv2 = np.zeros((16, 16, 24), dtype=np.float32)
#stage2基本单元3-分支2-1x1普通卷积核2，BN参数
s2c3_b2_w_conv2 = np.zeros((24, 24, 1, 1), dtype=np.float32)
S2c3_b2_w_conv2 = readbinfile("./data/stage2_3_branch2_5_weight.bin", 24*24*1*1).reshape((24, 24, 1, 1))
S2c3_b2_bn_mean_conv2 = readbinfile("./data/stage2_3_branch2_6_running_mean.bin", 24)
S2c3_b2_bn_val_conv2 = readbinfile("./data/stage2_3_branch2_6_running_var.bin", 24)
S2c3_b2_bn_gamma_conv2 = readbinfile("./data/stage2_3_branch2_6_weight.bin", 24)
S2c3_b2_bn_beta_conv2 = readbinfile("./data/stage2_3_branch2_6_bias.bin", 24)
#stage2基本单元3-通道合并，通道混洗
in_s2c3_shuff = np.zeros((16, 16, 48), dtype=np.float32)
out_s2c3_shuff = np.zeros((16, 16, 48), dtype=np.float32)

#####################################################
#stage3下采样单元-分支1-3x3深度卷积输出
out_s3s_b1_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3下采样单元-分支1-3x3深度卷积核，BN参数
#stage3下采样单元-分支1-1x1普通卷积输出
out_s3s_b1_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3下采样单元-分支1-1x1普通卷积核，BN参数

#stage3下采样单元-分支2-1x1普通卷积1输出
out_s3s_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3下采样单元-分支2-1x1普通卷积1核，BN参数
#stage3下采样单元-分支2-3x3深度卷积输出
out_s3s_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3下采样单元-分支2-3x3深度卷积核，BN参数
#stage3下采样单元-分支2-1x1普通卷积输出2
out_s3s_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3下采样单元-分支2-1x1普通卷积核2，BN参数

#stage3下采样单元-通道合并，通道混洗
in_s3s_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3s_shuff = np.zeros((8, 8, 96), dtype=np.float32)


#基本单元1
#stage3基本单元1-通道拆分
out_s3c1_ch_spilt1 = np.zeros((8, 8, 48), dtype=np.float32)
out_s3c1_ch_spilt2 = np.zeros((8, 8, 48), dtype=np.float32)

#stage3基本单元1-分支1

#stage3基本单元1-分支2-1x1普通卷积1输出
out_s3c1_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元1-分支2-1x1普通卷积1核，BN参数
#stage3基本单元1-分支2-3x3深度卷积输出
out_s3c1_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元1-分支2-3x3深度卷积核，BN参数
#stage3基本单元1-分支2-1x1普通卷积输出2
out_s3c1_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元1-分支2-1x1普通卷积核2，BN参数

#stage3基本单元1-通道合并，通道混洗
in_s3c1_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3c1_shuff = np.zeros((8, 8, 96), dtype=np.float32)


#基本单元2
#stage3基本单元2-通道拆分
out_s3c2_ch_spilt1 = np.zeros((8, 8, 48), dtype=np.float32)
out_s3c2_ch_spilt2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元2-分支1
#stage3基本单元2-分支2-1x1普通卷积1输出
out_s3c2_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元2-分支2-1x1普通卷积1核，BN参数
#stage3基本单元2-分支2-3x3深度卷积输出
out_s3c2_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元2-分支2-3x3深度卷积核，BN参数)
#stage3基本单元2-分支2-1x1普通卷积输出2
out_s3c2_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元2-分支2-1x1普通卷积核2，BN参数
#stage3基本单元2-通道合并，通道混洗
in_s3c2_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3c2_shuff = np.zeros((8, 8, 96), dtype=np.float32)


#基本单元3
#stage3基本单元3-通道拆分
out_s3c3_ch_spilt1 = np.zeros((8, 8, 48), dtype=np.float32)
out_s3c3_ch_spilt2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元3-分支1
#stage3基本单元3-分支2-1x1普通卷积1输出
out_s3c3_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元3-分支2-1x1普通卷积1核，BN参数
#stage3基本单元3-分支2-3x3深度卷积输出
out_s3c3_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元3-分支2-3x3深度卷积核，BN参数
#stage3基本单元3-分支2-1x1普通卷积输出2
out_s3c3_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元3-分支2-1x1普通卷积核2，BN参数
#stage3基本单元3-通道合并，通道混洗
in_s3c3_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3c3_shuff = np.zeros((8, 8, 96), dtype=np.float32)


#基本单元4
#stage3基本单元4-通道拆分
out_s3c4_ch_spilt1 = np.zeros((8, 8, 48), dtype=np.float32)
out_s3c4_ch_spilt2 = np.zeros((8, 8, 48), dtype=np.float32)  

#stage3基本单元4-分支1
#stage3基本单元4-分支2-1x1普通卷积1输出
out_s3c4_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元4-分支2-1x1普通卷积1核，BN参数
#stage3基本单元4-分支2-3x3深度卷积输出
out_s3c4_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元4-分支2-3x3深度卷积核，BN参数
#stage3基本单元4-分支2-1x1普通卷积输出2
out_s3c4_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元4-分支2-1x1普通卷积核2，BN参数
#stage3基本单元4-通道合并，通道混洗
in_s3c4_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3c4_shuff = np.zeros((8, 8, 96), dtype=np.float32)


#基本单元5
#stage3基本单元5-通道拆分
out_s3c5_ch_spilt1 = np.zeros((8, 8, 48), dtype=np.float32)
out_s3c5_ch_spilt2 = np.zeros((8, 8, 48), dtype=np.float32) 
#stage3基本单元5-分支1
#stage3基本单元5-分支2-1x1普通卷积1输出
out_s3c5_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元5-分支2-1x1普通卷积1核，BN参数
#stage3基本单元5-分支2-3x3深度卷积输出
out_s3c5_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元5-分支2-3x3深度卷积核，BN参数
#stage3基本单元5-分支2-1x1普通卷积输出2
out_s3c5_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元5-分支2-1x1普通卷积核2，BN参数
#stage3基本单元5-通道合并，通道混洗
in_s3c5_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3c5_shuff = np.zeros((8, 8, 96), dtype=np.float32)

#基本单元6
#stage3基本单元6-通道拆分
out_s3c6_ch_spilt1 = np.zeros((8, 8, 48), dtype=np.float32)
out_s3c6_ch_spilt2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元6-分支1
#stage3基本单元6-分支2-1x1普通卷积1输出
out_s3c6_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元6-分支2-1x1普通卷积1核，BN参数
#stage3基本单元6-分支2-3x3深度卷积输出
out_s3c6_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元6-分支2-3x3深度卷积核，BN参数
#stage3基本单元6-分支2-1x1普通卷积输出2
out_s3c6_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元6-分支2-1x1普通卷积核2，BN参数
#stage3基本单元6-通道合并，通道混洗
in_s3c6_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3c6_shuff = np.zeros((8, 8, 96), dtype=np.float32)

#基本单元7
#stage3基本单元7-通道拆分
out_s3c7_ch_spilt1 = np.zeros((8, 8, 48), dtype=np.float32)
out_s3c7_ch_spilt2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元7-分支1
#stage3基本单元7-分支2-1x1普通卷积1输出
out_s3c7_b2_conv1 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元7-分支2-1x1普通卷积1核，BN参数
#stage3基本单元7-分支2-3x3深度卷积输出
out_s3c7_b2_convdw = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元7-分支2-3x3深度卷积核，BN参数
#stage3基本单元7-分支2-1x1普通卷积输出2
out_s3c7_b2_conv2 = np.zeros((8, 8, 48), dtype=np.float32)
#stage3基本单元7-分支2-1x1普通卷积核2，BN参数
#stage3基本单元7-通道合并，通道混洗
in_s3c7_shuff = np.zeros((8, 8, 96), dtype=np.float32)
out_s3c7_shuff = np.zeros((8, 8, 96), dtype=np.float32)

print("Stage3_Sample:\tloading weight...\n ")
S3s_b1_w_convdw = readbinfile("./data/stage3_0_branch1_0_weight.bin", 48*3*3).reshape((48, 3, 3))
S3s_b1_bn_mean_convdw = readbinfile("./data/stage3_0_branch1_1_running_mean.bin", 48)
S3s_b1_bn_val_convdw = readbinfile("./data/stage3_0_branch1_1_running_var.bin", 48)
S3s_b1_bn_gamma_convdw = readbinfile("./data/stage3_0_branch1_1_weight.bin", 48)
S3s_b1_bn_beta_convdw = readbinfile("./data/stage3_0_branch1_1_bias.bin", 48)

S3s_b1_w_conv1 = readbinfile("./data/stage3_0_branch1_2_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3s_b1_bn_mean_conv1 = readbinfile("./data/stage3_0_branch1_3_running_mean.bin", 48)
S3s_b1_bn_val_conv1 = readbinfile("./data/stage3_0_branch1_3_running_var.bin", 48)
S3s_b1_bn_gamma_conv1 = readbinfile("./data/stage3_0_branch1_3_weight.bin", 48)
S3s_b1_bn_beta_conv1 = readbinfile("./data/stage3_0_branch1_3_bias.bin", 48)

S3s_b2_w_conv1 = readbinfile("./data/stage3_0_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3s_b2_bn_mean_conv1 = readbinfile("./data/stage3_0_branch2_1_running_mean.bin", 48)
S3s_b2_bn_val_conv1 = readbinfile("./data/stage3_0_branch2_1_running_var.bin", 48)
S3s_b2_bn_gamma_conv1 = readbinfile("./data/stage3_0_branch2_1_weight.bin", 48)
S3s_b2_bn_beta_conv1 = readbinfile("./data/stage3_0_branch2_1_bias.bin", 48)

S3s_b2_w_convdw = readbinfile("./data/stage3_0_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3s_b2_bn_mean_convdw = readbinfile("./data/stage3_0_branch2_4_running_mean.bin", 48)
S3s_b2_bn_val_convdw = readbinfile("./data/stage3_0_branch2_4_running_var.bin", 48)
S3s_b2_bn_gamma_convdw = readbinfile("./data/stage3_0_branch2_4_weight.bin", 48)
S3s_b2_bn_beta_convdw = readbinfile("./data/stage3_0_branch2_4_bias.bin", 48)

S3s_b2_w_conv2 = readbinfile("./data/stage3_0_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3s_b2_bn_mean_conv2 = readbinfile("./data/stage3_0_branch2_6_running_mean.bin", 48)
S3s_b2_bn_val_conv2 = readbinfile("./data/stage3_0_branch2_6_running_var.bin", 48)
S3s_b2_bn_gamma_conv2 = readbinfile("./data/stage3_0_branch2_6_weight.bin", 48)
S3s_b2_bn_beta_conv2 = readbinfile("./data/stage3_0_branch2_6_bias.bin", 48)


print("Stage3_Common1:\tloading weight... \n")
S3c1_b2_w_conv1 = readbinfile("./data/stage3_1_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c1_b2_bn_mean_conv1 = readbinfile("./data/stage3_1_branch2_1_running_mean.bin", 48)
S3c1_b2_bn_val_conv1 = readbinfile("./data/stage3_1_branch2_1_running_var.bin", 48)
S3c1_b2_bn_gamma_conv1 = readbinfile("./data/stage3_1_branch2_1_weight.bin", 48)
S3c1_b2_bn_beta_conv1 = readbinfile("./data/stage3_1_branch2_1_bias.bin", 48)

S3c1_b2_w_convdw = readbinfile("./data/stage3_1_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3c1_b2_bn_mean_convdw = readbinfile("./data/stage3_1_branch2_4_running_mean.bin", 48)
S3c1_b2_bn_val_convdw = readbinfile("./data/stage3_1_branch2_4_running_var.bin", 48)
S3c1_b2_bn_gamma_convdw = readbinfile("./data/stage3_1_branch2_4_weight.bin", 48)
S3c1_b2_bn_beta_convdw = readbinfile("./data/stage3_1_branch2_4_bias.bin", 48)

S3c1_b2_w_conv2 = readbinfile("./data/stage3_1_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c1_b2_bn_mean_conv2 = readbinfile("./data/stage3_1_branch2_6_running_mean.bin", 48)
S3c1_b2_bn_val_conv2 = readbinfile("./data/stage3_1_branch2_6_running_var.bin", 48)
S3c1_b2_bn_gamma_conv2 = readbinfile("./data/stage3_1_branch2_6_weight.bin", 48)
S3c1_b2_bn_beta_conv2 = readbinfile("./data/stage3_1_branch2_6_bias.bin", 48)


print("Stage3_Common2:\tloading weight... \n")
S3c2_b2_w_conv1 = readbinfile("./data/stage3_2_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c2_b2_bn_mean_conv1 = readbinfile("./data/stage3_2_branch2_1_running_mean.bin", 48)
S3c2_b2_bn_val_conv1 = readbinfile("./data/stage3_2_branch2_1_running_var.bin", 48)
S3c2_b2_bn_gamma_conv1 = readbinfile("./data/stage3_2_branch2_1_weight.bin", 48)
S3c2_b2_bn_beta_conv1 = readbinfile("./data/stage3_2_branch2_1_bias.bin", 48)

S3c2_b2_w_convdw = readbinfile("./data/stage3_2_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3c2_b2_bn_mean_convdw = readbinfile("./data/stage3_2_branch2_4_running_mean.bin", 48)
S3c2_b2_bn_val_convdw = readbinfile("./data/stage3_2_branch2_4_running_var.bin", 48)
S3c2_b2_bn_gamma_convdw = readbinfile("./data/stage3_2_branch2_4_weight.bin", 48)
S3c2_b2_bn_beta_convdw = readbinfile("./data/stage3_2_branch2_4_bias.bin", 48)

S3c2_b2_w_conv2 = readbinfile("./data/stage3_2_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c2_b2_bn_mean_conv2 = readbinfile("./data/stage3_2_branch2_6_running_mean.bin", 48)
S3c2_b2_bn_val_conv2 = readbinfile("./data/stage3_2_branch2_6_running_var.bin", 48)
S3c2_b2_bn_gamma_conv2 = readbinfile("./data/stage3_2_branch2_6_weight.bin", 48)
S3c2_b2_bn_beta_conv2 = readbinfile("./data/stage3_2_branch2_6_bias.bin", 48)


print("Stage3_Common3:\tloading weight... \n")
S3c3_b2_w_conv1 = readbinfile("./data/stage3_3_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c3_b2_bn_mean_conv1 = readbinfile("./data/stage3_3_branch2_1_running_mean.bin", 48)
S3c3_b2_bn_val_conv1 = readbinfile("./data/stage3_3_branch2_1_running_var.bin", 48)
S3c3_b2_bn_gamma_conv1 = readbinfile("./data/stage3_3_branch2_1_weight.bin", 48)
S3c3_b2_bn_beta_conv1 = readbinfile("./data/stage3_3_branch2_1_bias.bin", 48)

S3c3_b2_w_convdw = readbinfile("./data/stage3_3_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3c3_b2_bn_mean_convdw = readbinfile("./data/stage3_3_branch2_4_running_mean.bin", 48)
S3c3_b2_bn_val_convdw = readbinfile("./data/stage3_3_branch2_4_running_var.bin", 48)
S3c3_b2_bn_gamma_convdw = readbinfile("./data/stage3_3_branch2_4_weight.bin", 48)
S3c3_b2_bn_beta_convdw = readbinfile("./data/stage3_3_branch2_4_bias.bin", 48)

S3c3_b2_w_conv2 = readbinfile("./data/stage3_3_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c3_b2_bn_mean_conv2 = readbinfile("./data/stage3_3_branch2_6_running_mean.bin", 48)
S3c3_b2_bn_val_conv2 = readbinfile("./data/stage3_3_branch2_6_running_var.bin", 48)
S3c3_b2_bn_gamma_conv2 = readbinfile("./data/stage3_3_branch2_6_weight.bin", 48)
S3c3_b2_bn_beta_conv2 = readbinfile("./data/stage3_3_branch2_6_bias.bin", 48)


print("Stage3_Common4:\tloading weight... \n")
S3c4_b2_w_conv1 = readbinfile("./data/stage3_4_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c4_b2_bn_mean_conv1 = readbinfile("./data/stage3_4_branch2_1_running_mean.bin", 48)
S3c4_b2_bn_val_conv1 = readbinfile("./data/stage3_4_branch2_1_running_var.bin", 48)
S3c4_b2_bn_gamma_conv1 = readbinfile("./data/stage3_4_branch2_1_weight.bin", 48)
S3c4_b2_bn_beta_conv1 = readbinfile("./data/stage3_4_branch2_1_bias.bin", 48)

S3c4_b2_w_convdw = readbinfile("./data/stage3_4_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3c4_b2_bn_mean_convdw = readbinfile("./data/stage3_4_branch2_4_running_mean.bin", 48)
S3c4_b2_bn_val_convdw = readbinfile("./data/stage3_4_branch2_4_running_var.bin", 48)
S3c4_b2_bn_gamma_convdw = readbinfile("./data/stage3_4_branch2_4_weight.bin", 48)
S3c4_b2_bn_beta_convdw = readbinfile("./data/stage3_4_branch2_4_bias.bin", 48)

S3c4_b2_w_conv2 = readbinfile("./data/stage3_4_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c4_b2_bn_mean_conv2 = readbinfile("./data/stage3_4_branch2_6_running_mean.bin", 48)
S3c4_b2_bn_val_conv2 = readbinfile("./data/stage3_4_branch2_6_running_var.bin", 48)
S3c4_b2_bn_gamma_conv2 = readbinfile("./data/stage3_4_branch2_6_weight.bin", 48)
S3c4_b2_bn_beta_conv2 = readbinfile("./data/stage3_4_branch2_6_bias.bin", 48)


print("Stage3_Common5:\tloading weight... \n")
S3c5_b2_w_conv1 = readbinfile("./data/stage3_5_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c5_b2_bn_mean_conv1 = readbinfile("./data/stage3_5_branch2_1_running_mean.bin", 48)
S3c5_b2_bn_val_conv1 = readbinfile("./data/stage3_5_branch2_1_running_var.bin", 48)
S3c5_b2_bn_gamma_conv1 = readbinfile("./data/stage3_5_branch2_1_weight.bin", 48)
S3c5_b2_bn_beta_conv1 = readbinfile("./data/stage3_5_branch2_1_bias.bin", 48)

S3c5_b2_w_convdw = readbinfile("./data/stage3_5_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3c5_b2_bn_mean_convdw = readbinfile("./data/stage3_5_branch2_4_running_mean.bin", 48)
S3c5_b2_bn_val_convdw = readbinfile("./data/stage3_5_branch2_4_running_var.bin", 48)
S3c5_b2_bn_gamma_convdw = readbinfile("./data/stage3_5_branch2_4_weight.bin", 48)
S3c5_b2_bn_beta_convdw = readbinfile("./data/stage3_5_branch2_4_bias.bin", 48)

S3c5_b2_w_conv2 = readbinfile("./data/stage3_5_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c5_b2_bn_mean_conv2 = readbinfile("./data/stage3_5_branch2_6_running_mean.bin", 48)
S3c5_b2_bn_val_conv2 = readbinfile("./data/stage3_5_branch2_6_running_var.bin", 48)
S3c5_b2_bn_gamma_conv2 = readbinfile("./data/stage3_5_branch2_6_weight.bin", 48)
S3c5_b2_bn_beta_conv2 = readbinfile("./data/stage3_5_branch2_6_bias.bin", 48)


print("Stage3_Common6:\tloading weight... \n")
S3c6_b2_w_conv1 = readbinfile("./data/stage3_6_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c6_b2_bn_mean_conv1 = readbinfile("./data/stage3_6_branch2_1_running_mean.bin", 48)
S3c6_b2_bn_val_conv1 = readbinfile("./data/stage3_6_branch2_1_running_var.bin", 48)
S3c6_b2_bn_gamma_conv1 = readbinfile("./data/stage3_6_branch2_1_weight.bin", 48)
S3c6_b2_bn_beta_conv1 = readbinfile("./data/stage3_6_branch2_1_bias.bin", 48)

S3c6_b2_w_convdw = readbinfile("./data/stage3_6_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3c6_b2_bn_mean_convdw = readbinfile("./data/stage3_6_branch2_4_running_mean.bin", 48)
S3c6_b2_bn_val_convdw = readbinfile("./data/stage3_6_branch2_4_running_var.bin", 48)
S3c6_b2_bn_gamma_convdw = readbinfile("./data/stage3_6_branch2_4_weight.bin", 48)
S3c6_b2_bn_beta_convdw = readbinfile("./data/stage3_6_branch2_4_bias.bin", 48)

S3c6_b2_w_conv2 = readbinfile("./data/stage3_6_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c6_b2_bn_mean_conv2 = readbinfile("./data/stage3_6_branch2_6_running_mean.bin", 48)
S3c6_b2_bn_val_conv2 = readbinfile("./data/stage3_6_branch2_6_running_var.bin", 48)
S3c6_b2_bn_gamma_conv2 = readbinfile("./data/stage3_6_branch2_6_weight.bin", 48)
S3c6_b2_bn_beta_conv2 = readbinfile("./data/stage3_6_branch2_6_bias.bin", 48)


print("Stage3_Common7:\tloading weight...\n ")
S3c7_b2_w_conv1 = readbinfile("./data/stage3_7_branch2_0_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c7_b2_bn_mean_conv1 = readbinfile("./data/stage3_7_branch2_1_running_mean.bin", 48)
S3c7_b2_bn_val_conv1 = readbinfile("./data/stage3_7_branch2_1_running_var.bin", 48)
S3c7_b2_bn_gamma_conv1 = readbinfile("./data/stage3_7_branch2_1_weight.bin", 48)
S3c7_b2_bn_beta_conv1 = readbinfile("./data/stage3_7_branch2_1_bias.bin", 48)

S3c7_b2_w_convdw = readbinfile("./data/stage3_7_branch2_3_weight.bin", 48*3*3).reshape((48, 3, 3))
S3c7_b2_bn_mean_convdw = readbinfile("./data/stage3_7_branch2_4_running_mean.bin", 48)
S3c7_b2_bn_val_convdw = readbinfile("./data/stage3_7_branch2_4_running_var.bin", 48)
S3c7_b2_bn_gamma_convdw = readbinfile("./data/stage3_7_branch2_4_weight.bin", 48)
S3c7_b2_bn_beta_convdw = readbinfile("./data/stage3_7_branch2_4_bias.bin", 48)

S3c7_b2_w_conv2 = readbinfile("./data/stage3_7_branch2_5_weight.bin", 48*48*1*1).reshape((48, 48, 1, 1))
S3c7_b2_bn_mean_conv2 = readbinfile("./data/stage3_7_branch2_6_running_mean.bin", 48)
S3c7_b2_bn_val_conv2 = readbinfile("./data/stage3_7_branch2_6_running_var.bin", 48)
S3c7_b2_bn_gamma_conv2 = readbinfile("./data/stage3_7_branch2_6_weight.bin", 48)
S3c7_b2_bn_beta_conv2 = readbinfile("./data/stage3_7_branch2_6_bias.bin", 48)
#####################################################
#stage4下采样单元-分支1-3x3深度卷积输出
out_s4s_b1_convdw = np.zeros((4, 4, 96), dtype=np.float32)
#stage4下采样单元-分支1-3x3深度卷积核，BN参数
S4s_b1_w_convdw = readbinfile("./data/stage4_0_branch1_0_weight.bin", 96*3*3).reshape((96, 3, 3))
S4s_b1_bn_mean_convdw = readbinfile("./data/stage4_0_branch1_1_running_mean.bin", 96)
S4s_b1_bn_val_convdw = readbinfile("./data/stage4_0_branch1_1_running_var.bin", 96)
S4s_b1_bn_gamma_convdw = readbinfile("./data/stage4_0_branch1_1_weight.bin", 96)
S4s_b1_bn_beta_convdw = readbinfile("./data/stage4_0_branch1_1_bias.bin", 96)
#stage4下采样单元-分支1-1x1普通卷积输出
out_s4s_b1_conv1 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4下采样单元-分支1-1x1普通卷积核，BN参数
S4s_b1_w_conv1 = readbinfile("./data/stage4_0_branch1_2_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4s_b1_bn_mean_conv1 = readbinfile("./data/stage4_0_branch1_3_running_mean.bin", 96)
S4s_b1_bn_val_conv1 = readbinfile("./data/stage4_0_branch1_3_running_var.bin", 96)
S4s_b1_bn_gamma_conv1 = readbinfile("./data/stage4_0_branch1_3_weight.bin", 96)
S4s_b1_bn_beta_conv1 = readbinfile("./data/stage4_0_branch1_3_bias.bin", 96)
#stage4下采样单元-分支2-1x1普通卷积1输出
out_s4s_b2_conv1 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4下采样单元-分支2-1x1普通卷积1核，BN参数
S4s_b2_w_conv1 = readbinfile("./data/stage4_0_branch2_0_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4s_b2_bn_mean_conv1 = readbinfile("./data/stage4_0_branch2_1_running_mean.bin", 96)
S4s_b2_bn_val_conv1 = readbinfile("./data/stage4_0_branch2_1_running_var.bin", 96)
S4s_b2_bn_gamma_conv1 = readbinfile("./data/stage4_0_branch2_1_weight.bin", 96)
S4s_b2_bn_beta_conv1 = readbinfile("./data/stage4_0_branch2_1_bias.bin", 96)

#stage4下采样单元-分支2-3x3深度卷积输出
out_s4s_b2_convdw = np.zeros((4, 4, 96), dtype=np.float32)
#stage4下采样单元-分支2-3x3深度卷积核，BN参数
S4s_b2_w_convdw = readbinfile("./data/stage4_0_branch2_3_weight.bin", 96*3*3).reshape((96, 3, 3))
S4s_b2_bn_mean_convdw = readbinfile("./data/stage4_0_branch2_4_running_mean.bin", 96)
S4s_b2_bn_val_convdw = readbinfile("./data/stage4_0_branch2_4_running_var.bin", 96)
S4s_b2_bn_gamma_convdw = readbinfile("./data/stage4_0_branch2_4_weight.bin", 96)
S4s_b2_bn_beta_convdw = readbinfile("./data/stage4_0_branch2_4_bias.bin", 96)
#stage4下采样单元-分支2-1x1普通卷积输出2
out_s4s_b2_conv2 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4下采样单元-分支2-1x1普通卷积核2，BN参数
S4s_b2_w_conv2 = readbinfile("./data/stage4_0_branch2_5_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4s_b2_bn_mean_conv2 = readbinfile("./data/stage4_0_branch2_6_running_mean.bin", 96)
S4s_b2_bn_val_conv2 = readbinfile("./data/stage4_0_branch2_6_running_var.bin", 96)
S4s_b2_bn_gamma_conv2 = readbinfile("./data/stage4_0_branch2_6_weight.bin", 96)
S4s_b2_bn_beta_conv2 = readbinfile("./data/stage4_0_branch2_6_bias.bin", 96)
#stage4下采样单元-通道合并，通道混洗
in_s4s_shuff = np.zeros((4, 4, 192), dtype=np.float32)
out_s4s_shuff = np.zeros((4, 4, 192), dtype=np.float32)


#基本单元1
#stage4基本单元1-通道拆分
out_s4c1_ch_spilt1 = np.zeros((4, 4, 96), dtype=np.float32)
out_s4c1_ch_spilt2 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元1-分支1
#stage4基本单元1-分支2-1x1普通卷积1输出
out_s4c1_b2_conv1 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元1-分支2-1x1普通卷积1核，BN参数
S4c1_b2_w_conv1 = readbinfile("./data/stage4_1_branch2_0_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4c1_b2_bn_mean_conv1 = readbinfile("./data/stage4_1_branch2_1_running_mean.bin", 96)
S4c1_b2_bn_val_conv1 = readbinfile("./data/stage4_1_branch2_1_running_var.bin", 96)
S4c1_b2_bn_gamma_conv1 = readbinfile("./data/stage4_1_branch2_1_weight.bin", 96)
S4c1_b2_bn_beta_conv1 = readbinfile("./data/stage4_1_branch2_1_bias.bin", 96)
#stage4基本单元1-分支2-3x3深度卷积输出
out_s4c1_b2_convdw = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元1-分支2-3x3深度卷积核，BN参数
S4c1_b2_w_convdw = readbinfile("./data/stage4_1_branch2_3_weight.bin", 96*3*3).reshape((96, 3, 3))
S4c1_b2_bn_mean_convdw = readbinfile("./data/stage4_1_branch2_4_running_mean.bin", 96)
S4c1_b2_bn_val_convdw = readbinfile("./data/stage4_1_branch2_4_running_var.bin", 96)
S4c1_b2_bn_gamma_convdw = readbinfile("./data/stage4_1_branch2_4_weight.bin", 96)
S4c1_b2_bn_beta_convdw = readbinfile("./data/stage4_1_branch2_4_bias.bin", 96)
#stage4基本单元1-分支2-1x1普通卷积输出2
out_s4c1_b2_conv2 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元1-分支2-1x1普通卷积核2，BN参数
S4c1_b2_w_conv2 = readbinfile("./data/stage4_1_branch2_5_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4c1_b2_bn_mean_conv2 = readbinfile("./data/stage4_1_branch2_6_running_mean.bin", 96)
S4c1_b2_bn_val_conv2 = readbinfile("./data/stage4_1_branch2_6_running_var.bin", 96)
S4c1_b2_bn_gamma_conv2 = readbinfile("./data/stage4_1_branch2_6_weight.bin", 96)
S4c1_b2_bn_beta_conv2 = readbinfile("./data/stage4_1_branch2_6_bias.bin", 96)

#stage4基本单元1-通道合并，通道混洗
in_s4c1_shuff = np.zeros((4, 4, 192), dtype=np.float32)
out_s4c1_shuff = np.zeros((4, 4, 192), dtype=np.float32)


#基本单元2
#stage4基本单元2-通道拆分
out_s4c2_ch_spilt1 = np.zeros((4, 4, 96), dtype=np.float32)
out_s4c2_ch_spilt2 = np.zeros((4, 4, 96), dtype=np.float32)

#stage4基本单元2-分支1

#stage4基本单元2-分支2-1x1普通卷积1输出
out_s4c2_b2_conv1 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元2-分支2-1x1普通卷积1核，BN参数
S4c2_b2_w_conv1 = readbinfile("./data/stage4_2_branch2_0_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4c2_b2_bn_mean_conv1 = readbinfile("./data/stage4_2_branch2_1_running_mean.bin", 96)
S4c2_b2_bn_val_conv1 = readbinfile("./data/stage4_2_branch2_1_running_var.bin", 96)
S4c2_b2_bn_gamma_conv1 = readbinfile("./data/stage4_2_branch2_1_weight.bin", 96)
S4c2_b2_bn_beta_conv1 = readbinfile("./data/stage4_2_branch2_1_bias.bin", 96)
#stage4基本单元2-分支2-3x3深度卷积输出
out_s4c2_b2_convdw = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元2-分支2-3x3深度卷积核，BN参数
S4c2_b2_w_convdw = readbinfile("./data/stage4_2_branch2_3_weight.bin", 96*3*3).reshape((96, 3, 3))
S4c2_b2_bn_mean_convdw = readbinfile("./data/stage4_2_branch2_4_running_mean.bin", 96)
S4c2_b2_bn_val_convdw = readbinfile("./data/stage4_2_branch2_4_running_var.bin", 96)
S4c2_b2_bn_gamma_convdw = readbinfile("./data/stage4_2_branch2_4_weight.bin", 96)
S4c2_b2_bn_beta_convdw = readbinfile("./data/stage4_2_branch2_4_bias.bin", 96)
#stage4基本单元2-分支2-1x1普通卷积输出2
out_s4c2_b2_conv2 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元2-分支2-1x1普通卷积核2，BN参数
S4c2_b2_w_conv2 = readbinfile("./data/stage4_2_branch2_5_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4c2_b2_bn_mean_conv2 = readbinfile("./data/stage4_2_branch2_6_running_mean.bin", 96)
S4c2_b2_bn_val_conv2 = readbinfile("./data/stage4_2_branch2_6_running_var.bin", 96)
S4c2_b2_bn_gamma_conv2 = readbinfile("./data/stage4_2_branch2_6_weight.bin", 96)
S4c2_b2_bn_beta_conv2 = readbinfile("./data/stage4_2_branch2_6_bias.bin", 96)
#stage4基本单元2-通道合并，通道混洗
in_s4c2_shuff = np.zeros((4, 4, 192), dtype=np.float32)
out_s4c2_shuff = np.zeros((4, 4, 192), dtype=np.float32)

#基本单元3
#stage4基本单元3-通道拆分
out_s4c3_ch_spilt1 = np.zeros((4, 4, 96), dtype=np.float32)
out_s4c3_ch_spilt2 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元3-分支1
#stage4基本单元3-分支2-1x1普通卷积1输出
out_s4c3_b2_conv1 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元3-分支2-1x1普通卷积1核，BN参数
S4c3_b2_w_conv1 = readbinfile("./data/stage4_3_branch2_0_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4c3_b2_bn_mean_conv1 = readbinfile("./data/stage4_3_branch2_1_running_mean.bin", 96)
S4c3_b2_bn_val_conv1 = readbinfile("./data/stage4_3_branch2_1_running_var.bin", 96)
S4c3_b2_bn_gamma_conv1 = readbinfile("./data/stage4_3_branch2_1_weight.bin", 96)
S4c3_b2_bn_beta_conv1 = readbinfile("./data/stage4_3_branch2_1_bias.bin", 96)
#stage4基本单元3-分支2-3x3深度卷积输出
out_s4c3_b2_convdw = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元3-分支2-3x3深度卷积核，BN参数
S4c3_b2_w_convdw = readbinfile("./data/stage4_3_branch2_3_weight.bin", 96*3*3).reshape((96, 3, 3))
S4c3_b2_bn_mean_convdw = readbinfile("./data/stage4_3_branch2_4_running_mean.bin", 96)
S4c3_b2_bn_val_convdw = readbinfile("./data/stage4_3_branch2_4_running_var.bin", 96)
S4c3_b2_bn_gamma_convdw = readbinfile("./data/stage4_3_branch2_4_weight.bin", 96)
S4c3_b2_bn_beta_convdw = readbinfile("./data/stage4_3_branch2_4_bias.bin", 96)
#stage4基本单元3-分支2-1x1普通卷积输出2
out_s4c3_b2_conv2 = np.zeros((4, 4, 96), dtype=np.float32)
#stage4基本单元3-分支2-1x1普通卷积核2，BN参数
S4c3_b2_w_conv2 = readbinfile("./data/stage4_3_branch2_5_weight.bin", 96*96*1*1).reshape((96, 96, 1, 1))
S4c3_b2_bn_mean_conv2 = readbinfile("./data/stage4_3_branch2_6_running_mean.bin", 96)
S4c3_b2_bn_val_conv2 = readbinfile("./data/stage4_3_branch2_6_running_var.bin", 96)
S4c3_b2_bn_gamma_conv2 = readbinfile("./data/stage4_3_branch2_6_weight.bin", 96)
S4c3_b2_bn_beta_conv2 = readbinfile("./data/stage4_3_branch2_6_bias.bin", 96)
#stage4基本单元3-通道合并，通道混洗
in_s4c3_shuff = np.zeros((4, 4, 192), dtype=np.float32)
out_s4c3_shuff = np.zeros((4, 4, 192), dtype=np.float32)
###########################################################################################
#Conv5卷积核，BN参数
W_conv5 = readbinfile("./data/conv5_0_weight.bin", 1024*192*1*1).reshape((1024, 192, 1, 1))
BN_mean_conv5 = readbinfile("./data/conv5_1_running_mean.bin", 1024)
BN_val_conv5 = readbinfile("./data/conv5_1_running_var.bin", 1024)
BN_gamma_conv5 = readbinfile("./data/conv5_1_weight.bin", 1024)
BN_beta_conv5 = readbinfile("./data/conv5_1_bias.bin", 1024)
#Conv5输出4x4x1024
out_conv5 = np.zeros((4, 4, 1024), dtype=np.float32)
#Globalpool输出1024
out_globalpool = np.zeros((1024), dtype=np.float32)
#全连接层输出5(对应多少类)
out_fc = np.zeros((3), dtype=np.float32)
#全连接层权重参数
W_fc = readbinfile("./data/fc_weight.bin", 3*1024).reshape((3, 1024))
B_fc = readbinfile("./data/fc_bias.bin", 3)
# 加载权重后，立即打印关键信息
W_fc = readbinfile("./data/fc_weight.bin", 3*1024).reshape((3, 1024))
B_fc = readbinfile("./data/fc_bias.bin", 3)

# 验证W_fc
print("W_fc形状：", W_fc.shape)  # 必须输出 (4, 1024)
print("W_fc总元素数：", W_fc.size)  # 必须输出 4096
print("W_fc数据类型：", W_fc.dtype)  # 建议为 float32 或 float64（与输入向量一致）

# 验证B_fc
print("B_fc长度：", len(B_fc) if isinstance(B_fc, list) else B_fc.size)  # 必须输出 4

Stage3_Sample:	loading weight...
 
Stage3_Common1:	loading weight... 

Stage3_Common2:	loading weight... 

Stage3_Common3:	loading weight... 

Stage3_Common4:	loading weight... 

Stage3_Common5:	loading weight... 

Stage3_Common6:	loading weight... 

Stage3_Common7:	loading weight...
 
W_fc形状： (3, 1024)
W_fc总元素数： 3072
W_fc数据类型： float64
B_fc长度： 3


## 4. 软件推导

In [5]:
# image1 = cv2.imread(r"D:\7020\shuffle\data\2.jpg")
# pt0 = time.clock()
image1 = cv2.imread("./data01/2.jpg").astype(np.float32)
# image1_rgb = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB).astype(np.float32)
# #image1 = cv2.imread("./data/1.jpg", cv2.IMREAD_GRAYSCALE).astype(np.float32)

# mean = np.array([0.485, 0.456, 0.406])
# std = np.array([0.229, 0.224, 0.225])
# image = (image1_rgb / 255.0 - mean) / std
# print("Finish reading image.")
for r in range(128):
    for c in range(128):
        for ch in range(3):
            if ch == 2:
                # 第0通道（OpenCV默认B通道）：ImageNet标准化
                image[r][c][0] = ((image1[r][c][ch]/255) - 0.485)/0.229
            if ch == 1:
                # 第1通道（OpenCV默认G通道）：ImageNet标准化
                image[r][c][1] = ((image1[r][c][ch]/255) - 0.456)/0.224
            if ch == 0:
                # 第2通道（OpenCV默认R通道）：ImageNet标准化
                image[r][c][2] = ((image1[r][c][ch]/255) - 0.406)/0.225
print("Finish reading image.")
pt0 = time.time()
## Conv1
out_conv1 = sw_conv_bn_relu_layer(image, W_conv1, BN_mean_conv1, BN_val_conv1, BN_gamma_conv1, 
                                  BN_beta_conv1, 3, 2, 1, 0)
# print(out_conv1)
## maxpool
out_maxpool = sw_pooling(out_conv1, 24, 64, 64, 3, 2, 1, 0)
## stage2
# sample-branch1
out_s2s_b1_convdw = sw_conv_bn_relu_layer(out_maxpool, S2s_b1_w_convdw, S2s_b1_bn_mean_convdw,
                                         S2s_b1_bn_val_convdw, S2s_b1_bn_gamma_convdw,
                                         S2s_b1_bn_beta_convdw, 3, 2, 1, 1)
out_s2s_b1_conv1 = sw_conv_bn_relu_layer(out_s2s_b1_convdw, S2s_b1_w_conv1, S2s_b1_bn_mean_conv1,
                                         S2s_b1_bn_val_conv1, S2s_b1_bn_gamma_conv1,
                                         S2s_b1_bn_beta_conv1, 1, 1, 0, 0)
# sample-branch2
out_s2s_b2_conv1 = sw_conv_bn_relu_layer(out_maxpool, S2s_b2_w_conv1, S2s_b2_bn_mean_conv1,
                                         S2s_b2_bn_val_conv1, S2s_b2_bn_gamma_conv1,
                                         S2s_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s2s_b2_convdw = sw_conv_bn_relu_layer(out_s2s_b2_conv1, S2s_b2_w_convdw, S2s_b2_bn_mean_convdw,
                                         S2s_b2_bn_val_convdw, S2s_b2_bn_gamma_convdw,
                                         S2s_b2_bn_beta_convdw, 3, 2, 1, 1)
out_s2s_b2_conv2 = sw_conv_bn_relu_layer(out_s2s_b2_convdw, S2s_b2_w_conv2, S2s_b2_bn_mean_conv2,
                                         S2s_b2_bn_val_conv2, S2s_b2_bn_gamma_conv2,
                                         S2s_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s2s_shuff[:, :, :24] = out_s2s_b1_conv1
in_s2s_shuff[:, :, 24:] = out_s2s_b2_conv2
out_s2s_shuff = sw_channel_shuffle(in_s2s_shuff, 48, 16, 2)

# 基本单元1
out_s2c1_ch_spilt1[:, :, :] = out_s2s_shuff[:, :, :24]  # 前24通道
out_s2c1_ch_spilt2[:, :, :] = out_s2s_shuff[:, :, 24:]  # 后24通道
out_s2c1_b2_conv1 = sw_conv_bn_relu_layer(out_s2c1_ch_spilt2, S2c1_b2_w_conv1, S2c1_b2_bn_mean_conv1,
                                         S2c1_b2_bn_val_conv1, S2c1_b2_bn_gamma_conv1,
                                         S2c1_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s2c1_b2_convdw = sw_conv_bn_relu_layer(out_s2c1_b2_conv1, S2c1_b2_w_convdw, S2c1_b2_bn_mean_convdw,
                                         S2c1_b2_bn_val_convdw, S2c1_b2_bn_gamma_convdw,
                                         S2c1_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s2c1_b2_conv2 = sw_conv_bn_relu_layer(out_s2c1_b2_convdw, S2c1_b2_w_conv2, S2c1_b2_bn_mean_conv2,
                                         S2c1_b2_bn_val_conv2, S2c1_b2_bn_gamma_conv2,
                                         S2c1_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s2c1_shuff[:, :, :24] = out_s2c1_ch_spilt1
in_s2c1_shuff[:, :, 24:] = out_s2c1_b2_conv2
out_s2c1_shuff = sw_channel_shuffle(in_s2c1_shuff, 48, 16, 2)
# 基本单元2
out_s2c2_ch_spilt1[:, :, :] = out_s2c1_shuff[:, :, :24]  # 前24通道
out_s2c2_ch_spilt2[:, :, :] = out_s2c1_shuff[:, :, 24:]  # 后24通道
out_s2c2_b2_conv1 = sw_conv_bn_relu_layer(out_s2c2_ch_spilt2, S2c2_b2_w_conv1, S2c2_b2_bn_mean_conv1,
                                         S2c2_b2_bn_val_conv1, S2c2_b2_bn_gamma_conv1,
                                         S2c2_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s2c2_b2_convdw = sw_conv_bn_relu_layer(out_s2c2_b2_conv1, S2c2_b2_w_convdw, S2c2_b2_bn_mean_convdw,
                                         S2c2_b2_bn_val_convdw, S2c2_b2_bn_gamma_convdw,
                                         S2c2_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s2c2_b2_conv2 = sw_conv_bn_relu_layer(out_s2c2_b2_convdw, S2c2_b2_w_conv2, S2c2_b2_bn_mean_conv2,
                                         S2c2_b2_bn_val_conv2, S2c2_b2_bn_gamma_conv2,
                                         S2c2_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s2c2_shuff[:, :, :24] = out_s2c2_ch_spilt1
in_s2c2_shuff[:, :, 24:] = out_s2c2_b2_conv2
out_s2c2_shuff = sw_channel_shuffle(in_s2c2_shuff, 48, 16, 2)
# 基本单元3
out_s2c3_ch_spilt1[:, :, :] = out_s2c2_shuff[:, :, :24]  # 前24通道
out_s2c3_ch_spilt2[:, :, :] = out_s2c2_shuff[:, :, 24:]  # 后24通道
out_s2c3_b2_conv1 = sw_conv_bn_relu_layer(out_s2c3_ch_spilt2, S2c3_b2_w_conv1, S2c3_b2_bn_mean_conv1,
                                         S2c3_b2_bn_val_conv1, S2c3_b2_bn_gamma_conv1,
                                         S2c3_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s2c3_b2_convdw = sw_conv_bn_relu_layer(out_s2c3_b2_conv1, S2c3_b2_w_convdw, S2c3_b2_bn_mean_convdw,
                                         S2c3_b2_bn_val_convdw, S2c3_b2_bn_gamma_convdw,
                                         S2c3_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s2c3_b2_conv2 = sw_conv_bn_relu_layer(out_s2c3_b2_convdw, S2c3_b2_w_conv2, S2c3_b2_bn_mean_conv2,
                                         S2c3_b2_bn_val_conv2, S2c3_b2_bn_gamma_conv2,
                                         S2c3_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s2c3_shuff[:, :, :24] = out_s2c3_ch_spilt1
in_s2c3_shuff[:, :, 24:] = out_s2c3_b2_conv2
out_s2c3_shuff = sw_channel_shuffle(in_s2c3_shuff, 48, 16, 2)


## stage3
# sample-branch1
out_s3s_b1_convdw = sw_conv_bn_relu_layer(out_s2c3_shuff, S3s_b1_w_convdw, S3s_b1_bn_mean_convdw,
                                         S3s_b1_bn_val_convdw, S3s_b1_bn_gamma_convdw,
                                         S3s_b1_bn_beta_convdw, 3, 2, 1, 1)
out_s3s_b1_conv1 = sw_conv_bn_relu_layer(out_s3s_b1_convdw, S3s_b1_w_conv1, S3s_b1_bn_mean_conv1,
                                         S3s_b1_bn_val_conv1, S3s_b1_bn_gamma_conv1,
                                         S3s_b1_bn_beta_conv1, 1, 1, 0, 0)
# sample-branch2
out_s3s_b2_conv1 = sw_conv_bn_relu_layer(out_s2c3_shuff, S3s_b2_w_conv1, S3s_b2_bn_mean_conv1,
                                         S3s_b2_bn_val_conv1, S3s_b2_bn_gamma_conv1,
                                         S3s_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3s_b2_convdw = sw_conv_bn_relu_layer(out_s3s_b2_conv1, S3s_b2_w_convdw, S3s_b2_bn_mean_convdw,
                                         S3s_b2_bn_val_convdw, S3s_b2_bn_gamma_convdw,
                                         S3s_b2_bn_beta_convdw, 3, 2, 1, 1)
out_s3s_b2_conv2 = sw_conv_bn_relu_layer(out_s3s_b2_convdw, S3s_b2_w_conv2, S3s_b2_bn_mean_conv2,
                                         S3s_b2_bn_val_conv2, S3s_b2_bn_gamma_conv2,
                                         S3s_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3s_shuff[:, :, :48] = out_s3s_b1_conv1
in_s3s_shuff[:, :, 48:] = out_s3s_b2_conv2
out_s3s_shuff = sw_channel_shuffle(in_s3s_shuff, 96, 8, 2)
# 基本单元1
out_s3c1_ch_spilt1[:, :, :] = out_s3s_shuff[:, :, :48]  # 前24通道
out_s3c1_ch_spilt2[:, :, :] = out_s3s_shuff[:, :, 48:]  # 后24通道
out_s3c1_b2_conv1 = sw_conv_bn_relu_layer(out_s3c1_ch_spilt2, S3c1_b2_w_conv1, S3c1_b2_bn_mean_conv1,
                                         S3c1_b2_bn_val_conv1, S3c1_b2_bn_gamma_conv1,
                                         S3c1_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3c1_b2_convdw = sw_conv_bn_relu_layer(out_s3c1_b2_conv1, S3c1_b2_w_convdw, S3c1_b2_bn_mean_convdw,
                                         S3c1_b2_bn_val_convdw, S3c1_b2_bn_gamma_convdw,
                                         S3c1_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s3c1_b2_conv2 = sw_conv_bn_relu_layer(out_s3c1_b2_convdw, S3c1_b2_w_conv2, S3c1_b2_bn_mean_conv2,
                                         S3c1_b2_bn_val_conv2, S3c1_b2_bn_gamma_conv2,
                                         S3c1_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3c1_shuff[:, :, :48] = out_s3c1_ch_spilt1
in_s3c1_shuff[:, :, 48:] = out_s3c1_b2_conv2
out_s3c1_shuff = sw_channel_shuffle(in_s3c1_shuff, 96, 8, 2)
# 基本单元2
out_s3c2_ch_spilt1[:, :, :] = out_s3c1_shuff[:, :, :48]  # 前24通道
out_s3c2_ch_spilt2[:, :, :] = out_s3c1_shuff[:, :, 48:]  # 后24通道
out_s3c2_b2_conv1 = sw_conv_bn_relu_layer(out_s3c2_ch_spilt2, S3c2_b2_w_conv1, S3c2_b2_bn_mean_conv1,
                                         S3c2_b2_bn_val_conv1, S3c2_b2_bn_gamma_conv1,
                                         S3c2_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3c2_b2_convdw = sw_conv_bn_relu_layer(out_s3c2_b2_conv1, S3c2_b2_w_convdw, S3c2_b2_bn_mean_convdw,
                                         S3c2_b2_bn_val_convdw, S3c2_b2_bn_gamma_convdw,
                                         S3c2_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s3c2_b2_conv2 = sw_conv_bn_relu_layer(out_s3c2_b2_convdw, S3c2_b2_w_conv2, S3c2_b2_bn_mean_conv2,
                                         S3c2_b2_bn_val_conv2, S3c2_b2_bn_gamma_conv2,
                                         S3c2_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3c2_shuff[:, :, :48] = out_s3c2_ch_spilt1
in_s3c2_shuff[:, :, 48:] = out_s3c2_b2_conv2
out_s3c2_shuff = sw_channel_shuffle(in_s3c2_shuff, 96, 8, 2)
# 基本单元3
out_s3c3_ch_spilt1[:, :, :] = out_s3c2_shuff[:, :, :48]  # 前24通道
out_s3c3_ch_spilt2[:, :, :] = out_s3c2_shuff[:, :, 48:]  # 后24通道
out_s3c3_b2_conv1 = sw_conv_bn_relu_layer(out_s3c3_ch_spilt2, S3c3_b2_w_conv1, S3c3_b2_bn_mean_conv1,
                                         S3c3_b2_bn_val_conv1, S3c3_b2_bn_gamma_conv1,
                                         S3c3_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3c3_b2_convdw = sw_conv_bn_relu_layer(out_s3c3_b2_conv1, S3c3_b2_w_convdw, S3c3_b2_bn_mean_convdw,
                                         S3c3_b2_bn_val_convdw, S3c3_b2_bn_gamma_convdw,
                                         S3c3_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s3c3_b2_conv2 = sw_conv_bn_relu_layer(out_s3c3_b2_convdw, S3c3_b2_w_conv2, S3c3_b2_bn_mean_conv2,
                                         S3c3_b2_bn_val_conv2, S3c3_b2_bn_gamma_conv2,
                                         S3c3_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3c3_shuff[:, :, :48] = out_s3c3_ch_spilt1
in_s3c3_shuff[:, :, 48:] = out_s3c3_b2_conv2
out_s3c3_shuff = sw_channel_shuffle(in_s3c3_shuff, 96, 8, 2)
# 基本单元4
out_s3c4_ch_spilt1[:, :, :] = out_s3c3_shuff[:, :, :48]  # 前24通道
out_s3c4_ch_spilt2[:, :, :] = out_s3c3_shuff[:, :, 48:]  # 后24通道
out_s3c4_b2_conv1 = sw_conv_bn_relu_layer(out_s3c4_ch_spilt2, S3c4_b2_w_conv1, S3c4_b2_bn_mean_conv1,
                                         S3c4_b2_bn_val_conv1, S3c4_b2_bn_gamma_conv1,
                                         S3c4_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3c4_b2_convdw = sw_conv_bn_relu_layer(out_s3c4_b2_conv1, S3c4_b2_w_convdw, S3c4_b2_bn_mean_convdw,
                                         S3c4_b2_bn_val_convdw, S3c4_b2_bn_gamma_convdw,
                                         S3c4_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s3c4_b2_conv2 = sw_conv_bn_relu_layer(out_s3c4_b2_convdw, S3c4_b2_w_conv2, S3c4_b2_bn_mean_conv2,
                                         S3c4_b2_bn_val_conv2, S3c4_b2_bn_gamma_conv2,
                                         S3c4_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3c4_shuff[:, :, :48] = out_s3c4_ch_spilt1
in_s3c4_shuff[:, :, 48:] = out_s3c4_b2_conv2
out_s3c4_shuff = sw_channel_shuffle(in_s3c4_shuff, 96, 8, 2)
# 基本单元5
out_s3c5_ch_spilt1[:, :, :] = out_s3c4_shuff[:, :, :48]  # 前24通道
out_s3c5_ch_spilt2[:, :, :] = out_s3c4_shuff[:, :, 48:]  # 后24通道
out_s3c5_b2_conv1 = sw_conv_bn_relu_layer(out_s3c5_ch_spilt2, S3c5_b2_w_conv1, S3c5_b2_bn_mean_conv1,
                                         S3c5_b2_bn_val_conv1, S3c5_b2_bn_gamma_conv1,
                                         S3c5_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3c5_b2_convdw = sw_conv_bn_relu_layer(out_s3c5_b2_conv1, S3c5_b2_w_convdw, S3c5_b2_bn_mean_convdw,
                                         S3c5_b2_bn_val_convdw, S3c5_b2_bn_gamma_convdw,
                                         S3c5_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s3c5_b2_conv2 = sw_conv_bn_relu_layer(out_s3c5_b2_convdw, S3c5_b2_w_conv2, S3c5_b2_bn_mean_conv2,
                                         S3c5_b2_bn_val_conv2, S3c5_b2_bn_gamma_conv2,
                                         S3c5_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3c5_shuff[:, :, :48] = out_s3c5_ch_spilt1
in_s3c5_shuff[:, :, 48:] = out_s3c5_b2_conv2
out_s3c5_shuff = sw_channel_shuffle(in_s3c5_shuff, 96, 8, 2)
# 基本单元6
out_s3c6_ch_spilt1[:, :, :] = out_s3c5_shuff[:, :, :48]  # 前24通道
out_s3c6_ch_spilt2[:, :, :] = out_s3c5_shuff[:, :, 48:]  # 后24通道
out_s3c6_b2_conv1 = sw_conv_bn_relu_layer(out_s3c6_ch_spilt2, S3c6_b2_w_conv1, S3c6_b2_bn_mean_conv1,
                                         S3c6_b2_bn_val_conv1, S3c6_b2_bn_gamma_conv1,
                                         S3c6_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3c6_b2_convdw = sw_conv_bn_relu_layer(out_s3c6_b2_conv1, S3c6_b2_w_convdw, S3c6_b2_bn_mean_convdw,
                                         S3c6_b2_bn_val_convdw, S3c6_b2_bn_gamma_convdw,
                                         S3c6_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s3c6_b2_conv2 = sw_conv_bn_relu_layer(out_s3c6_b2_convdw, S3c6_b2_w_conv2, S3c6_b2_bn_mean_conv2,
                                         S3c6_b2_bn_val_conv2, S3c6_b2_bn_gamma_conv2,
                                         S3c6_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3c6_shuff[:, :, :48] = out_s3c6_ch_spilt1
in_s3c6_shuff[:, :, 48:] = out_s3c6_b2_conv2
out_s3c6_shuff = sw_channel_shuffle(in_s3c6_shuff, 96, 8, 2)
# 基本单元7
out_s3c7_ch_spilt1[:, :, :] = out_s3c6_shuff[:, :, :48]  # 前24通道
out_s3c7_ch_spilt2[:, :, :] = out_s3c6_shuff[:, :, 48:]  # 后24通道
out_s3c7_b2_conv1 = sw_conv_bn_relu_layer(out_s3c7_ch_spilt2, S3c7_b2_w_conv1, S3c7_b2_bn_mean_conv1,
                                         S3c7_b2_bn_val_conv1, S3c7_b2_bn_gamma_conv1,
                                         S3c7_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s3c7_b2_convdw = sw_conv_bn_relu_layer(out_s3c7_b2_conv1, S3c7_b2_w_convdw, S3c7_b2_bn_mean_convdw,
                                         S3c7_b2_bn_val_convdw, S3c7_b2_bn_gamma_convdw,
                                         S3c7_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s3c7_b2_conv2 = sw_conv_bn_relu_layer(out_s3c7_b2_convdw, S3c7_b2_w_conv2, S3c7_b2_bn_mean_conv2,
                                         S3c7_b2_bn_val_conv2, S3c7_b2_bn_gamma_conv2,
                                         S3c7_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s3c7_shuff[:, :, :48] = out_s3c7_ch_spilt1
in_s3c7_shuff[:, :, 48:] = out_s3c7_b2_conv2
out_s3c7_shuff = sw_channel_shuffle(in_s3c7_shuff, 96, 8, 2)

## stage4
# sample-branch1
out_s4s_b1_convdw = sw_conv_bn_relu_layer(out_s3c7_shuff, S4s_b1_w_convdw, S4s_b1_bn_mean_convdw,
                                         S4s_b1_bn_val_convdw, S4s_b1_bn_gamma_convdw,
                                         S4s_b1_bn_beta_convdw, 3, 2, 1, 1)
out_s4s_b1_conv1 = sw_conv_bn_relu_layer(out_s4s_b1_convdw, S4s_b1_w_conv1, S4s_b1_bn_mean_conv1,
                                         S4s_b1_bn_val_conv1, S4s_b1_bn_gamma_conv1,
                                         S4s_b1_bn_beta_conv1, 1, 1, 0, 0)
# sample-branch2
out_s4s_b2_conv1 = sw_conv_bn_relu_layer(out_s3c7_shuff, S4s_b2_w_conv1, S4s_b2_bn_mean_conv1,
                                         S4s_b2_bn_val_conv1, S4s_b2_bn_gamma_conv1,
                                         S4s_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s4s_b2_convdw = sw_conv_bn_relu_layer(out_s4s_b2_conv1, S4s_b2_w_convdw, S4s_b2_bn_mean_convdw,
                                         S4s_b2_bn_val_convdw, S4s_b2_bn_gamma_convdw,
                                         S4s_b2_bn_beta_convdw, 3, 2, 1, 1)
out_s4s_b2_conv2 = sw_conv_bn_relu_layer(out_s4s_b2_convdw, S4s_b2_w_conv2, S4s_b2_bn_mean_conv2,
                                         S4s_b2_bn_val_conv2, S4s_b2_bn_gamma_conv2,
                                         S4s_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s4s_shuff[:, :, :96] = out_s4s_b1_conv1
in_s4s_shuff[:, :, 96:] = out_s4s_b2_conv2
out_s4s_shuff = sw_channel_shuffle(in_s4s_shuff, 192, 4, 2)
# 基本单元1
out_s4c1_ch_spilt1[:, :, :] = out_s4s_shuff[:, :, :96]  # 前24通道
out_s4c1_ch_spilt2[:, :, :] = out_s4s_shuff[:, :, 96:]  # 后24通道
out_s4c1_b2_conv1 = sw_conv_bn_relu_layer(out_s4c1_ch_spilt2, S4c1_b2_w_conv1, S4c1_b2_bn_mean_conv1,
                                         S4c1_b2_bn_val_conv1, S4c1_b2_bn_gamma_conv1,
                                         S4c1_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s4c1_b2_convdw = sw_conv_bn_relu_layer(out_s4c1_b2_conv1, S4c1_b2_w_convdw, S4c1_b2_bn_mean_convdw,
                                         S4c1_b2_bn_val_convdw, S4c1_b2_bn_gamma_convdw,
                                         S4c1_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s4c1_b2_conv2 = sw_conv_bn_relu_layer(out_s4c1_b2_convdw, S4c1_b2_w_conv2, S4c1_b2_bn_mean_conv2,
                                         S4c1_b2_bn_val_conv2, S4c1_b2_bn_gamma_conv2,
                                         S4c1_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s4c1_shuff[:, :, :96] = out_s4c1_ch_spilt1
in_s4c1_shuff[:, :, 96:] = out_s4c1_b2_conv2
out_s4c1_shuff = sw_channel_shuffle(in_s4c1_shuff, 192, 4, 2)
# print("out_s4c1_b2_conv2\n")
# print(out_s4c1_b2_conv2)
# 基本单元2
out_s4c2_ch_spilt1[:, :, :] = out_s4c1_shuff[:, :, :96]  # 前24通道
out_s4c2_ch_spilt2[:, :, :] = out_s4c1_shuff[:, :, 96:]  # 后24通道
out_s4c2_b2_conv1 = sw_conv_bn_relu_layer(out_s4c2_ch_spilt2, S4c2_b2_w_conv1, S4c2_b2_bn_mean_conv1,
                                         S4c2_b2_bn_val_conv1, S4c2_b2_bn_gamma_conv1,
                                         S4c2_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s4c2_b2_convdw = sw_conv_bn_relu_layer(out_s4c2_b2_conv1, S4c2_b2_w_convdw, S4c2_b2_bn_mean_convdw,
                                         S4c2_b2_bn_val_convdw, S4c2_b2_bn_gamma_convdw,
                                         S4c2_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s4c2_b2_conv2 = sw_conv_bn_relu_layer(out_s4c2_b2_convdw, S4c2_b2_w_conv2, S4c2_b2_bn_mean_conv2,
                                         S4c2_b2_bn_val_conv2, S4c2_b2_bn_gamma_conv2,
                                         S4c2_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s4c2_shuff[:, :, :96] = out_s4c2_ch_spilt1
in_s4c2_shuff[:, :, 96:] = out_s4c2_b2_conv2
out_s4c2_shuff = sw_channel_shuffle(in_s4c2_shuff, 192, 4, 2)
# 基本单元3
out_s4c3_ch_spilt1[:, :, :] = out_s4c2_shuff[:, :, :96]  # 前24通道
out_s4c3_ch_spilt2[:, :, :] = out_s4c2_shuff[:, :, 96:]  # 后24通道
out_s4c3_b2_conv1 = sw_conv_bn_relu_layer(out_s4c3_ch_spilt2, S4c3_b2_w_conv1, S4c3_b2_bn_mean_conv1,
                                         S4c3_b2_bn_val_conv1, S4c3_b2_bn_gamma_conv1,
                                         S4c3_b2_bn_beta_conv1, 1, 1, 0, 0)
out_s4c3_b2_convdw = sw_conv_bn_relu_layer(out_s4c3_b2_conv1, S4c3_b2_w_convdw, S4c3_b2_bn_mean_convdw,
                                         S4c3_b2_bn_val_convdw, S4c3_b2_bn_gamma_convdw,
                                         S4c3_b2_bn_beta_convdw, 3, 1, 1, 1)
out_s4c3_b2_conv2 = sw_conv_bn_relu_layer(out_s4c3_b2_convdw, S4c3_b2_w_conv2, S4c3_b2_bn_mean_conv2,
                                         S4c3_b2_bn_val_conv2, S4c3_b2_bn_gamma_conv2,
                                         S4c3_b2_bn_beta_conv2, 1, 1, 0, 0)
in_s4c3_shuff[:, :, :96] = out_s4c3_ch_spilt1
in_s4c3_shuff[:, :, 96:] = out_s4c3_b2_conv2
out_s4c3_shuff = sw_channel_shuffle(in_s4c3_shuff, 192, 4, 2)

## conv5
out_conv5 = sw_conv_bn_relu_layer(out_s4c3_shuff, W_conv5, BN_mean_conv5,
                                         BN_val_conv5, BN_gamma_conv5,
                                         BN_beta_conv5, 1, 1, 0, 0)
## Pool
out_globalpool = sw_pooling(out_conv5, 1024, 4, 4, 4, 1, 0, 1)
# print("全局池化后扁平化向量长度：", out_globalpool_flat.size)  # 预期输出 1024
## fc
print(out_globalpool.shape)
out_globalpool_flat = out_globalpool.flatten()
# print("全局池化后扁平化向量长度：", out_globalpool_flat.size)  # 预期输出 1024
out_fc = sw_fully_connected(1024, 3, out_globalpool_flat, W_fc, B_fc)
# print("全连接层输出形状：", out_fc.shape)  # 预期输出 (4,)
# print("全连接层输出示例：", out_fc)  # 查看具体数值是否合理（无异常NaN/Inf）
# print(out_fc.shape)
max_val = np.max (out_fc)
exp_x = np.exp (out_fc - max_val) # 防数值溢出
sum_exp = np.sum (exp_x)
out_final = exp_x /sum_exp
print(out_final)
pt1 = time.time()
time_sw = pt1 - pt0
print("Software inference done.")

print("Time consumed: %fs" % time_sw)



Finish reading image.
(1, 1, 1024)
[  9.77045953e-01   1.74106262e-05   2.29366552e-02]
Software inference done.
Time consumed: 898.101895s
